In [18]:
# %pip install -r requirements.txt

## Import libraries

In [19]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset, load_from_disk
from collections import Counter
import pandas as pd
import numpy as np

import datasets
datasets.logging.set_verbosity_error()

## Load datasets

In [20]:
review_dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_Electronics", split="full", trust_remote_code=True)

## Data preprocessing

### Review dataset

Check dataset features

In [21]:
review_dataset

Dataset({
    features: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase'],
    num_rows: 43886944
})

Remove unwanted or uninformative columns

In [22]:
review_dataset = review_dataset.remove_columns(['asin', 'images', 'timestamp', 'helpful_vote'])

In [23]:
print(review_dataset.unique('rating'))
print(review_dataset.unique('verified_purchase'))

[3.0, 1.0, 5.0, 4.0, 2.0, 0.0]
[True, False]


We can filter the dataset to exclude reviews with 0.0 rating and unverified_purchases

In [24]:
review_dataset = review_dataset.filter(lambda x: x['verified_purchase'] == True and x['rating'] != 0, num_proc=10)

In [25]:
review_dataset = review_dataset.remove_columns(['verified_purchase'])

In [26]:
review_dataset.save_to_disk('datasets/review_dataset_attrfiltered', num_proc=10)

Saving the dataset (27/27 shards): 100%|██████████| 40546882/40546882 [06:52<00:00, 98403.37 examples/s] 


### Item dataset

In [27]:
item_dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Electronics", split="full", trust_remote_code=True)

In [28]:
item_dataset

Dataset({
    features: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author'],
    num_rows: 1610012
})

In [29]:
item_dataset[4]

{'main_category': 'Cell Phones & Accessories',
 'title': 'Motorola Droid X Essentials Combo Pack',
 'average_rating': 3.8,
 'rating_number': 64,
 'features': ['New Droid X Essentials Combo Pack',
  'Exclusive Package Incredible Value Worth $145!!!',
  'Includes all Genuine High Quality Motorola Made Accessories'],
 'description': ['all Genuine High Quality Motorola Made Accessories, including Multimedia Station with HDMI technology, HDMI Cable and AC Wall Charger, Motorola Navigation / Music Vehicle Charging Mount Car Dock and Motorola 12v Vehicle Power Adapter Charger!'],
 'price': '14.99',
 'images': {'hi_res': [None, None, None, None, None],
  'large': ['https://m.media-amazon.com/images/I/51-DXSMlHaL._AC_.jpg',
   'https://m.media-amazon.com/images/I/31HYf51qCQL._AC_.jpg',
   'https://m.media-amazon.com/images/I/31P1+fPxUNL._AC_.jpg',
   'https://m.media-amazon.com/images/I/31nLJFuQiXL._AC_.jpg',
   'https://m.media-amazon.com/images/I/21DyMQ6Bk5L._AC_.jpg'],
  'thumb': ['https://m

Remove unwanted columns

In [30]:
item_dataset = item_dataset.remove_columns(['main_category',
                            'categories',
                            'images',       # Remove if needed
                            'bought_together', 
                            'store', 
                            'subtitle', 
                            'author', 
                            'videos',
                            'details'])

In [31]:
item_dataset[4]

{'title': 'Motorola Droid X Essentials Combo Pack',
 'average_rating': 3.8,
 'rating_number': 64,
 'features': ['New Droid X Essentials Combo Pack',
  'Exclusive Package Incredible Value Worth $145!!!',
  'Includes all Genuine High Quality Motorola Made Accessories'],
 'description': ['all Genuine High Quality Motorola Made Accessories, including Multimedia Station with HDMI technology, HDMI Cable and AC Wall Charger, Motorola Navigation / Music Vehicle Charging Mount Car Dock and Motorola 12v Vehicle Power Adapter Charger!'],
 'price': '14.99',
 'parent_asin': 'B004E2Z88O'}

In [32]:
item_dataset.save_to_disk('datasets/item_dataset_attrfiltered', num_proc=10)

Saving the dataset (10/10 shards): 100%|██████████| 1610012/1610012 [00:13<00:00, 115419.00 examples/s]


## Attr filtered dataset

In [34]:
filtered_review_dataset = load_from_disk('datasets/review_dataset_attrfiltered')
filtered_review_dataset

Dataset({
    features: ['rating', 'title', 'text', 'parent_asin', 'user_id'],
    num_rows: 40546882
})

In [35]:
filtered_item_dataset = load_from_disk('datasets/item_dataset_attrfiltered')
filtered_item_dataset

Dataset({
    features: ['title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'parent_asin'],
    num_rows: 1610012
})

Update rating numbers

In [36]:
asin_counts = Counter(filtered_review_dataset['parent_asin'])

filtered_item_dataset = filtered_item_dataset.map(lambda x, counts=asin_counts: {'rating_number': counts.get(x['parent_asin'], 0)}, num_proc=10)

Map (num_proc=10): 100%|██████████| 1610012/1610012 [01:43<00:00, 15563.09 examples/s] 


Filter products without price or less than 50 reviews

In [37]:
min_rating = 30
filtered_item_dataset = filtered_item_dataset.filter(lambda x, m=min_rating: x['rating_number'] >= m and x['price'] is not None, num_proc=10)

Filter (num_proc=10): 100%|██████████| 1610012/1610012 [00:12<00:00, 126752.46 examples/s]


Compare with item dataset to get rid of reviews of products that aren't in dataset

In [38]:
print(len(filtered_item_dataset.unique('parent_asin')))
print(len(filtered_review_dataset.unique('parent_asin')))

Flattening the indices: 100%|██████████| 166942/166942 [00:10<00:00, 15520.43 examples/s]


166942
1535611


In [39]:
missing_asins = set(filtered_review_dataset.unique('parent_asin')) - set(filtered_item_dataset.unique('parent_asin'))
print(len(missing_asins))

1368669


In [40]:
filtered_review_dataset = filtered_review_dataset.filter(lambda x, S=missing_asins: x['parent_asin'] not in S, num_proc=10)

Filter (num_proc=10): 100%|██████████| 40546882/40546882 [01:45<00:00, 385499.30 examples/s]  


In [42]:
filtered_review_dataset.save_to_disk('datasets/review_dataset_asinfiltered', num_proc=10)

Saving the dataset (21/21 shards): 100%|██████████| 34105969/34105969 [02:10<00:00, 262223.30 examples/s]


In [43]:
filtered_item_dataset.save_to_disk('datasets/item_dataset_asinfiltered', num_proc=10)

Saving the dataset (10/10 shards): 100%|██████████| 166942/166942 [00:06<00:00, 27271.44 examples/s]


## Asin filtered dataset

In [44]:
asin_review_dataset = load_from_disk('datasets/review_dataset_asinfiltered')
asin_review_dataset

Dataset({
    features: ['rating', 'title', 'text', 'parent_asin', 'user_id'],
    num_rows: 34105969
})

In [45]:
asin_item_dataset = load_from_disk('datasets/item_dataset_asinfiltered')
asin_item_dataset

Dataset({
    features: ['title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'parent_asin'],
    num_rows: 166942
})

Remove reviews made by users with 3 reviews or less

In [46]:
#Remove reviews made by users with 5 reviews or less
min_reviews_per_user = 5
user_counts = Counter(asin_review_dataset['user_id'])

valid_user_ids = {user_id for user_id, count in user_counts.items() if count > min_reviews_per_user}

In [47]:
asin_review_dataset = asin_review_dataset.filter(lambda x, S=valid_user_ids: x['user_id'] in S, num_proc=10)

Filter (num_proc=10): 100%|██████████| 34105969/34105969 [01:36<00:00, 352207.84 examples/s] 


In [48]:
asin_review_dataset

Dataset({
    features: ['rating', 'title', 'text', 'parent_asin', 'user_id'],
    num_rows: 9928825
})

Recount ratings and averages

In [49]:
pd_df = asin_review_dataset.select_columns(['parent_asin', 'rating']).to_pandas()
stats = pd_df.groupby('parent_asin')['rating'].agg(['mean', 'count'])
stats['mean'] = stats['mean'].round(2)

mean_stats = stats['mean'].to_dict()
count_stats = stats['count'].to_dict()


Check for missing products

In [ ]:
missing_asins = set(asin_item_dataset.unique('parent_asin')) - set(asin_review_dataset.unique('parent_asin'))

asin_item_dataset = asin_item_dataset.filter(lambda x, S=missing_asins: x['parent_asin'] not in S, num_proc=10)

Filter (num_proc=10): 100%|██████████| 166942/166942 [00:08<00:00, 19295.01 examples/s]


In [53]:
def update_numbers(x, mean_stats=mean_stats, count_stats=count_stats):
    idx = x['parent_asin']
    x['rating_number'] = count_stats.get(idx,0)
    x['average_rating'] = mean_stats.get(idx,0)
    return x

asin_item_dataset = asin_item_dataset.map(update_numbers, num_proc=10)

Map (num_proc=10): 100%|██████████| 166703/166703 [00:19<00:00, 8340.52 examples/s] 


In [54]:
asin_item_dataset.save_to_disk('datasets/item_dataset_final', num_proc=10)
asin_review_dataset.save_to_disk('datasets/review_dataset_final', num_proc=10)

Saving the dataset (10/10 shards): 100%|██████████| 9928825/9928825 [00:44<00:00, 222474.63 examples/s]


## Final filtered dataset

In [56]:
final_review_dataset = load_from_disk('datasets/review_dataset_final')
final_item_dataset = load_from_disk('datasets/item_dataset_final')

In [57]:
final_item_dataset

Dataset({
    features: ['title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'parent_asin'],
    num_rows: 166703
})

Select the top 2000 with more ratings

In [ ]:
final_item_dataset=final_item_dataset.sort('rating_number', reverse=True).select(range(2000))

{'title': 'EarFun Wireless Earbuds, [2020 CES Award] Free Bluetooth 5.0 Earbuds with Wireless Charging Case, USB-C Quick Charge, IPX7 Waterproof in-Ear Wireless Headphones, Deep Bass, 30H Playtime',
 'average_rating': 4.24,
 'rating_number': 573,
 'features': ['HI-FI STEREO SOUND: Feature the dual graphene speakers that produce incredible sound quality with deep bass and crystal clear treble; built-in MEMS microphones with noise cancellation for you to enjoy crystal-clear hands-free calls and voice assistant in noisy circumstances',
  'TWS & BLUETOOTH 5.0: Powered by latest Bluetooth 5.0 which dramatically enhances the stability and connection of the earbuds; after a one-time setup, the bluetooth earbuds will automatically turn on and pair with your smart device; EarFun Free Bluetooth Earphones work flawlessly at a distance up to 49 feet (15 meters)',
  'IPX7 TRULY SWEAT & WATERPROOF: With innovative SweatShield Technology, EarFun Free Wireless Earphones can easily resist from sweat an

In [65]:
asin = set(final_item_dataset.unique('parent_asin'))
final_review_dataset = final_review_dataset.filter(lambda x, S=asin: x['parent_asin'] in S, num_proc=10)

Filter (num_proc=10): 100%|██████████| 9928825/9928825 [00:25<00:00, 389286.15 examples/s]


In [66]:
final_review_dataset

Dataset({
    features: ['rating', 'title', 'text', 'parent_asin', 'user_id'],
    num_rows: 2815510
})

In [67]:
final_review_dataset.save_to_disk('datasets/review_dataset_final_top2000', num_proc=10)
final_item_dataset.save_to_disk('datasets/item_dataset_final_top2000', num_proc=10)

Saving the dataset (10/10 shards): 100%|██████████| 2000/2000 [02:57<00:00, 11.27 examples/s]
